# 01 · Ingestão Bronze
Lê o `dataset.json` do volume e grava os registros bruto na tabela Delta
`workspace.fauna.bronze_registros`, com duas colunas de auditoria (de onde e quando o dado entrou).

In [0]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
print("Fuso da sessão:", spark.conf.get("spark.sql.session.timeZone"))

In [0]:
caminho_origem = "/Volumes/workspace/fauna/raw/dataset.json"

df_raw = (
    spark.read
    .option("multiLine", True)
    .json(caminho_origem)
)

df_raw.printSchema()
display(df_raw.limit(10))

In [0]:
from pyspark.sql import functions as F

colunas_originais = [
    "id_registro", "id_camera", "data_hora_inicio", "duracao_segundos",
    "especie", "grupo", "individuos", "temperatura_c", "umidade_pct",
]

df_bronze = df_raw.select(
    *colunas_originais,
    F.col("_metadata.file_path").alias("_arquivo_origem"),
    F.current_timestamp().alias("_data_ingestao"),
)

display(df_bronze.limit(5))

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.fauna.bronze_registros")
)

In [0]:
bronze = spark.table("workspace.fauna.bronze_registros")

print(f"Linhas : {bronze.count()}")
print(f"Colunas: {len(bronze.columns)}")

In [0]:
%sql
DESCRIBE HISTORY workspace.fauna.bronze_registros